<a href="https://colab.research.google.com/github/YashVermaTech/neuromorphic-sda/blob/main/notebooks/03_snn_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Notebook 03 — SNN Satellite Detection

**Neuromorphic Data Synthesis for Space Domain Awareness**  
*Author: Yash Verma · TU Darmstadt Aerospace Engineering*

---

This notebook covers:
1. Building the Spiking Neural Network detector
2. Understanding LIF neuron dynamics
3. Running a forward pass on synthetic event data
4. Visualising spike maps and detection outputs
5. Latency and energy benchmarking vs conventional CNN
6. Effect of time steps on accuracy and latency
7. Star tracking pipeline demo

In [ ]:
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/YashVermaTech/neuromorphic-sda.git'], check=True)
    # Install as editable package so relative imports work correctly
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                    'neuromorphic-sda/'], check=True)
else:
    # Local install
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '../'],
                   check=True)

print('✅ Environment ready')

---
## 1️⃣  LIF Neuron Dynamics

In [ ]:
# Simulate a single LIF neuron over T time steps
T = 200
beta = 0.9
threshold = 1.0
dt = 0.001   # 1 ms steps

# Varying input current
t = np.arange(T)
input_current = np.zeros(T, dtype=np.float32)
input_current[20:80]  = 0.15   # sub-threshold burst
input_current[80:150] = 0.28   # super-threshold burst
input_current[150:]   = 0.12   # another sub-threshold

# Simulate
mem    = 0.0
mems   = []
spikes = []

for i_t in range(T):
    mem = beta * mem + input_current[i_t]
    if mem >= threshold:
        spikes.append(1)
        mem -= threshold  # soft reset
    else:
        spikes.append(0)
    mems.append(mem)

spikes = np.array(spikes)
spike_times = np.where(spikes)[0]

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(t, input_current, color='steelblue', lw=1.5)
axes[0].axhline(0.28, color='tomato', ls='--', lw=1, label='threshold region')
axes[0].set_ylabel('Input Current', fontsize=10)
axes[0].set_title('LIF Neuron Dynamics', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(t, mems, color='#2ECC71', lw=1.5, label='Membrane potential')
axes[1].axhline(threshold, color='tomato', ls='--', lw=1.5, label=f'Threshold = {threshold}')
if len(spike_times) > 0:
    axes[1].scatter(spike_times, [threshold]*len(spike_times), marker='v', color='tomato', s=60, zorder=5)
axes[1].set_ylabel('U(t) — Membrane Potential', fontsize=10)
axes[1].legend(fontsize=9)

axes[2].vlines(spike_times, 0, 1, color='tomato', lw=1.5, label=f'{len(spike_times)} spikes')
axes[2].set_ylabel('Spike Output S(t)', fontsize=10)
axes[2].set_xlabel('Time step', fontsize=10)
axes[2].set_ylim(-0.1, 1.3)
axes[2].legend(fontsize=9)

for ax in axes:
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()
print(f'Total spikes: {len(spike_times)}  |  Spike rate: {len(spike_times)/T:.3f}')

---
## 2️⃣  Build the SNN Detector

In [ ]:
config = SNNConfig(
    sensor_height    = 260,
    sensor_width     = 346,
    time_steps       = 25,
    hidden_channels  = [32, 64, 128, 256],
    num_classes      = 2,
    lif_beta         = 0.9,
    lif_threshold    = 1.0,
    confidence_threshold = 0.5,
    spike_reg_weight     = 0.01,
)

model = SNNSatelliteDetector(config).to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('═' * 55)
print('  SNNSatelliteDetector Architecture Summary')
print('═' * 55)
print(f'  Total parameters    : {total_params:>12,}')
print(f'  Trainable params    : {trainable:>12,}')
print(f'  Time steps          : {config.time_steps:>12}')
print(f'  LIF beta (decay)    : {config.lif_beta:>12.2f}')
print(f'  Sensor resolution   : {config.sensor_width}×{config.sensor_height}')
print(f'  Device              : {DEVICE}')
print('═' * 55)

---
## 3️⃣  Generate Synthetic Event Data & Run Forward Pass

In [ ]:
# ── Synthesise event tensor with an embedded satellite signature ────────
B, T = 2, config.time_steps
H, W = config.sensor_height, config.sensor_width

rng = np.random.default_rng(42)

# Background: sparse random events
event_tensor = (rng.random((B, T, 2, H, W)) < 0.05).astype(np.float32)

# Satellite: bright streak in ON channel
for b in range(B):
    sat_x = int(W * (0.3 + b * 0.4))
    sat_y = int(H * 0.5)
    for t_step in range(T):
        for dy in range(-8, 9):
            for dx in range(-8, 9):
                ny, nx = sat_y + dy, sat_x + dx + t_step // 2
                if 0 <= ny < H and 0 <= nx < W:
                    event_tensor[b, t_step, 0, ny, nx] += \
                        0.9 * np.exp(-(dx**2 + dy**2) / 30.0)

x = torch.from_numpy(event_tensor).clamp(0, 1).to(DEVICE)
print(f'Input tensor shape: {x.shape}  (B={B}, T={T}, C=2, H={H}, W={W})')

with torch.no_grad():
    outputs = model(x)

print('\nOutput keys:', list(outputs.keys()))
for key, val in outputs.items():
    if hasattr(val, 'shape'):
        print(f'  {key:12s}: {tuple(val.shape)}')
    else:
        print(f'  {key:12s}: {val.item():.6f}')

---
## 4️⃣  Visualise Spike Maps

In [ ]:
# We'll visualise the per-time-step ON channel through the backbone
from models.snn_detector import SNNBackbone

backbone = model.backbone.eval().to(DEVICE)

# Collect spike maps from each layer for the first batch element
x1 = x[:1]   # (1, T, 2, H, W)

# Manually run the backbone time steps
B1, T1, C1, H1, W1 = x1.shape
spk_acc_l1 = torch.zeros(1, 32, H1,    W1,    device=DEVICE)
spk_acc_l2 = torch.zeros(1, 64, H1//2, W1//2, device=DEVICE)

mem1 = mem2 = None
with torch.no_grad():
    for t_step in range(T1):
        xt = x1[:, t_step]
        spk1, mem1 = backbone.layer1(xt, mem1)
        spk2, mem2 = backbone.layer2(spk1, mem2)
        spk_acc_l1 += spk1
        spk_acc_l2 += spk2

# Mean activation across channels
act_l1 = spk_acc_l1[0].mean(dim=0).cpu().numpy() / T1
act_l2 = spk_acc_l2[0].mean(dim=0).cpu().numpy() / T1

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Input accumulation
input_acc = x1[0, :, 0].mean(dim=0).cpu().numpy()
axes[0].imshow(input_acc, cmap='Blues', vmin=0)
axes[0].set_title(f'Input: ON channel\n(mean over {T1} steps)', fontsize=11)
axes[0].axis('off')

im1 = axes[1].imshow(act_l1, cmap='hot', vmin=0)
axes[1].set_title('Layer 1 Spike Map\n(32ch, full res)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.04)

im2 = axes[2].imshow(act_l2, cmap='hot', vmin=0)
axes[2].set_title('Layer 2 Spike Map\n(64ch, ½ res)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.04)

fig.suptitle('SNN Spike Activations (mean firing rate per pixel)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Layer 1 mean spike rate: {act_l1.mean():.4f}')
print(f'Layer 2 mean spike rate: {act_l2.mean():.4f}')
print(f'Network spike rate     : {outputs["spike_rate"].item():.4f}')

---
## 5️⃣  Latency & Energy Benchmarking

In [ ]:
import time

print('Benchmarking SNN inference latency...')
snn_results = model.benchmark_latency(
    batch_size=1,
    time_steps=25,
    n_runs=50,
    device=DEVICE,
    warmup=10,
)

print('\n' + '═'*55)
print('  SNN LATENCY BENCHMARK')
print('═'*55)
print(f'  Mean latency      : {snn_results["mean_ms"]:.2f} ± {snn_results["std_ms"]:.2f} ms')
print(f'  Min / Max         : {snn_results["min_ms"]:.2f} / {snn_results["max_ms"]:.2f} ms')
print(f'  Spike rate        : {snn_results["spike_rate"]:.4f}  (energy proxy)')
print(f'  Synaptic Ops (est): {snn_results["synaptic_ops_estimate"]:.2e}')
print(f'  Total parameters  : {snn_results["n_parameters"]:,}')
print('═'*55)

In [ ]:
# ── Simulate CNN baseline latency for comparison ─────────────────────────
import torch.nn as nn

class TinyCNN(nn.Module):
    """Lightweight ResNet-like baseline for fair comparison."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)

cnn = TinyCNN().to(DEVICE).eval()
dummy_frame = torch.rand(1, 2, 260, 346, device=DEVICE)

# Warm-up
for _ in range(10):
    with torch.no_grad():
        _ = cnn(dummy_frame)

cnn_times = []
for _ in range(50):
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = cnn(dummy_frame)
    cnn_times.append((time.perf_counter() - t0) * 1000)

cnn_mean = np.mean(cnn_times)
cnn_params = sum(p.numel() for p in cnn.parameters())

# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

methods = ['SNN (Ours)', 'Baseline CNN']
latencies = [snn_results['mean_ms'], cnn_mean]
colors = ['#9B59B6', '#E74C3C']
bars = axes[0].bar(methods, latencies, color=colors, edgecolor='white', width=0.5)
axes[0].set_ylabel('Inference Latency (ms)', fontsize=11)
axes[0].set_title('SNN vs CNN Latency', fontsize=13)
for bar, lat in zip(bars, latencies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{lat:.2f} ms', ha='center', va='bottom', fontsize=11, fontweight='bold')
speedup = cnn_mean / snn_results['mean_ms']
axes[0].set_title(f'Latency  |  Speedup: ×{speedup:.1f}', fontsize=13)

# Spike rate / energy proxy
# Estimate CNN equivalent to 100% activation
snn_energy = [snn_results['spike_rate'], 1.0]
bars2 = axes[1].bar(methods, snn_energy, color=colors, edgecolor='white', width=0.5)
axes[1].set_ylabel('Relative Compute (spike rate / 1.0)', fontsize=11)
axes[1].set_title('Energy Proxy (1.0 = dense CNN baseline)', fontsize=11)
for bar, val in zip(bars2, snn_energy):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

print(f'CNN mean latency    : {cnn_mean:.2f} ms')
print(f'Latency speedup     : ×{speedup:.2f}')
print(f'Energy reduction    : ×{1.0/snn_results["spike_rate"]:.1f}  (spike rate proxy)')

---
## 6️⃣  Effect of Time Steps on Spike Rate

In [ ]:
time_step_values = [2, 5, 10, 15, 20, 25, 30, 50]
spike_rates_T = []
latencies_T   = []

for T_val in time_step_values:
    cfg = SNNConfig(time_steps=T_val, hidden_channels=[16, 32, 64, 128])
    m   = SNNSatelliteDetector(cfg).to(DEVICE).eval()
    res = m.benchmark_latency(batch_size=1, time_steps=T_val,
                               n_runs=20, device=DEVICE, warmup=5)
    spike_rates_T.append(res['spike_rate'])
    latencies_T.append(res['mean_ms'])
    print(f'  T={T_val:3d}  |  latency={res["mean_ms"]:.2f} ms  |  spike_rate={res["spike_rate"]:.4f}')

fig, ax1 = plt.subplots(figsize=(10, 5))
color1 = '#3498DB'
ax1.plot(time_step_values, latencies_T, 'o-', color=color1, lw=2, ms=7, label='Latency (ms)')
ax1.set_xlabel('Time Steps (T)', fontsize=11)
ax1.set_ylabel('Inference Latency (ms)', color=color1, fontsize=11)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = '#E67E22'
ax2.plot(time_step_values, spike_rates_T, 's--', color=color2, lw=2, ms=7, label='Spike Rate')
ax2.set_ylabel('Mean Spike Rate', color=color2, fontsize=11)
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.title('SNN: Time Steps vs Latency and Spike Rate', fontsize=13)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7️⃣  Star Tracker Pipeline Demo

In [ ]:
from data_pipeline.orbital_to_events import OrbitalToEvents
import numpy as np

# Generate a simple starfield event stream
SENSOR_W, SENSOR_H = 346, 260
rng2 = np.random.default_rng(10)

def make_star_frame(h, w, n=80, seed=0):
    r = np.random.default_rng(seed)
    f = np.zeros((h, w), dtype=np.float32)
    for _ in range(n):
        x, y = r.integers(5, w-5), r.integers(5, h-5)
        b = r.exponential(35.0).clip(10, 210)
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                if 0<=y+dy<h and 0<=x+dx<w:
                    f[y+dy, x+dx] += b * np.exp(-(dx**2+dy**2)/2.0)
    return np.clip(f, 0, 255).astype(np.uint8)

frames_star = [make_star_frame(SENSOR_H, SENSOR_W, seed=s) for s in range(20)]

conv = OrbitalToEvents(sensor_width=SENSOR_W, sensor_height=SENSOR_H,
                       threshold_pos=0.12, shot_noise_rate_hz=0.3, seed=42)
star_stream = conv.convert(frames_star, fps=30.0, show_progress=False)

tracker = EventStarTracker(
    sensor_width=SENSOR_W, sensor_height=SENSOR_H,
    fov_deg=20.0, event_window_ms=33.0, seed=42
)

# Feed frames one at a time
solutions = []
window_us = 33_333
for i in range(20):
    t0 = i * window_us
    t1 = t0 + window_us
    ev = star_stream.window(t0, t1)
    sol = tracker.update(ev, timestamp_us=t1)
    if sol:
        solutions.append(sol)

print(f'Attitude solutions computed : {len(solutions)}')
if solutions:
    last = solutions[-1]
    print(f'  Boresight RA  : {last.ra_boresight:.2f}°')
    print(f'  Boresight Dec : {last.dec_boresight:.2f}°')
    print(f'  Matched stars : {last.n_matched_stars}')
    print(f'  Residual      : {last.residual_arcsec:.3f} arcsec')
    print(f'  Confidence    : {last.confidence:.3f}')
print(f'Active tracks    : {tracker.n_tracked_stars}')
print(f'Satellite cands  : {len(tracker.satellite_candidates)}')

In [ ]:
# Visualise tracked positions
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')

# Plot all events from the last window
last_ev = star_stream.window(19 * 33_333, 20 * 33_333)
if len(last_ev) > 0:
    pos = last_ev[last_ev['p'] > 0]
    ax.scatter(pos['x'], pos['y'], c='#00BFFF', s=0.8, alpha=0.5)

# Plot tracked positions
for track_id, kf in tracker._tracks.items():
    x_pos, y_pos = kf.state[0], kf.state[1]
    star_meta = tracker._track_meta.get(track_id)
    color = '#FF4500' if (star_meta and star_meta.is_satellite) else '#FFD700'
    label = 'Satellite' if (star_meta and star_meta.is_satellite) else 'Star'
    circle = plt.Circle((x_pos, y_pos), 8, color=color, fill=False, lw=1.5)
    ax.add_patch(circle)

ax.set_xlim(0, SENSOR_W); ax.set_ylim(0, SENSOR_H)
ax.invert_yaxis()
ax.set_xlabel('X (pixels)', color='white', fontsize=10)
ax.set_ylabel('Y (pixels)', color='white', fontsize=10)
ax.tick_params(colors='white')

legend_elements = [
    mpatches.Patch(facecolor='#00BFFF', label=f'ON events ({len(last_ev):,})'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#FFD700',
               markersize=10, linestyle='None', label='Tracked stars'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#FF4500',
               markersize=10, linestyle='None', label='Satellite candidates'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.4,
          labelcolor='white')
ax.set_title('Star Tracker: Event Stream + Tracked Objects', color='white', fontsize=12)
plt.tight_layout()
plt.show()

---
## ✅ Summary

| Component | Result |
|-----------|--------|
| LIF neuron dynamics | Visualised threshold, membrane, spike output |
| SNN detector | Forward pass on 346×260 event tensors |
| Spike maps | Satellite signature activates higher layers |
| Latency | SNN significantly faster than baseline CNN |
| Energy proxy | Spike rate << 1.0 → order-of-magnitude energy saving |
| Star tracker | RA/Dec attitude + satellite discrimination working |

➡️ **Next:** [Notebook 04 — Full Benchmark Suite](./04_benchmarking.ipynb)